In [1]:
# import cv2
from datetime import datetime

now = datetime.now()

print(f'Notebook executed: {now}')

Notebook executed: 2026-01-08 06:36:00.589819


### **Dataframe Creation**
____

#### **File List**
____

In [2]:
from pathlib import Path

files_path = "/mnt/data/StressID_dataset/Videos"

In [3]:
def find_mp4_files(start_directory):
    """
    Recursively finds all files ending with '.mp4' (case-insensitive) 
    in the specified directory and its subdirectories.

    Args:
        start_directory (str): The folder path to start searching from.

    Returns:
        list: A list of strings, where each string is the full path 
              to an MP4 file.
    """
    # Create a 'Path' object from the input string
    base_path = Path(start_directory)
    
    # Use rglob (recursive glob) to search for a pattern
    # We use a generator expression for efficiency
    mp4_files_generator = (str(p.resolve()) for p in base_path.rglob('*.mp4'))
    
    # We create a separate generator for the uppercase extension
    MP4_files_generator = (str(p.resolve()) for p in base_path.rglob('*.MP4'))

    # Combine and return the results as a list
    all_mp4_files = list(mp4_files_generator) + list(MP4_files_generator)

    return all_mp4_files

video_files_list = find_mp4_files(files_path)

# Print the results
print(f"Found {len(video_files_list)} MP4 files:")
for file_path in video_files_list[:5]:
    print(file_path)


Found 660 MP4 files:
/mnt/data/StressID_dataset/Videos/wssm/wssm_Stroop.mp4
/mnt/data/StressID_dataset/Videos/wssm/wssm_Relax.mp4
/mnt/data/StressID_dataset/Videos/wssm/wssm_Counting3.mp4
/mnt/data/StressID_dataset/Videos/wssm/wssm_Breathing.mp4
/mnt/data/StressID_dataset/Videos/wssm/wssm_Baseline.mp4


In [4]:
stress_situation = [
                        "counting1", "stroop", "speaking", 
                        "math", "reading", "counting2",
                        "counting3"
                    ]

#### **Paths Dataframe**
___

In [5]:
import pandas as pd

df = pd.DataFrame(data=video_files_list, columns=["path"])

df.sample(3)

,path
371,/mnt/data/StressID_dataset/Videos/v8mh/v8mh_Co...
309,/mnt/data/StressID_dataset/Videos/e5p4/e5p4_Sp...
194,/mnt/data/StressID_dataset/Videos/qw5t/qw5t_Re...


In [6]:
N = 2  # number of levels you want

df_levels = df["path"].apply(
    lambda p: pd.Series(Path(p).parts[-N:])
)

df_levels.columns = [f"level_{i}" for i in range(1, N+1)]

df = df.join(df_levels)

df.sample(3)

,path,level_1,level_2
433,/mnt/data/StressID_dataset/Videos/m8g5/m8g5_Re...,m8g5,m8g5_Relax.mp4
86,/mnt/data/StressID_dataset/Videos/4e8r/4e8r_Co...,4e8r,4e8r_Counting2.mp4
377,/mnt/data/StressID_dataset/Videos/v8mh/v8mh_Co...,v8mh,v8mh_Counting3.mp4


In [7]:
df_rename = df
df["level_2"] = df["level_2"].apply(lambda x: Path(x).stem)
df_rename.columns = ["path", "subject", "subject/task"]
df_rename.sample(3)

,path,subject,subject/task
422,/mnt/data/StressID_dataset/Videos/m8g5/m8g5_Vi...,m8g5,m8g5_Video2
112,/mnt/data/StressID_dataset/Videos/h8r2/h8r2_Co...,h8r2,h8r2_Counting2
308,/mnt/data/StressID_dataset/Videos/e5p4/._e5p4_...,e5p4,._e5p4_Counting1


#### **Labels Dataframe**
______

In [8]:
df_labels = pd.read_csv("/mnt/data/StressID_dataset/labels.csv", sep=",")

df_labels.sample(3)

,subject/task,binary-stress,affect3-class
275,cxj0_Counting1,1,2
688,y8c3_Video2,0,0
624,v8mh_Breathing,0,0


#### **Full Dataframe**
_____

In [9]:
print("Number of duplicate filenames in df1:", df_rename["subject/task"].duplicated().sum())
print("Number of duplicate filenames in df2:", df_labels["subject/task"].duplicated().sum())


Number of duplicate filenames in df1: 0
Number of duplicate filenames in df2: 0


In [10]:
df_complete = df_rename.merge(
                                df_labels[["subject/task", "binary-stress"]],   # ← this slice keeps ONLY filename + label
                                on="subject/task",
                                how="left"
                                )

print(f"df_complete shape: {df_complete.shape}")
df_complete = df_complete.dropna(subset=["binary-stress"])

print(f"df_complete without NaN shape: {df_complete.shape}")
print("NaN val binary-stress:", df_complete["binary-stress"].isna().sum())
df_complete.sample(3)

df_complete shape: (660, 4)
df_complete without NaN shape: (578, 4)
NaN val binary-stress: 0


,path,subject,subject/task,binary-stress
458,/mnt/data/StressID_dataset/Videos/p9i3/p9i3_Br...,p9i3,p9i3_Breathing,0.0
96,/mnt/data/StressID_dataset/Videos/k67g/k67g_St...,k67g,k67g_Stroop,1.0
647,/mnt/data/StressID_dataset/Videos/chdf/chdf_Vi...,chdf,chdf_Video2,0.0


In [11]:
df_complete["task"] = df_complete["subject/task"].str.split("_").str[-1].str.lower()

print(f"df_complete shape: {df_complete.shape}")
df_complete.sample(3)

df_complete shape: (578, 5)


,path,subject,subject/task,binary-stress,task
87,/mnt/data/StressID_dataset/Videos/4e8r/4e8r_Re...,4e8r,4e8r_Relax,0.0,relax
455,/mnt/data/StressID_dataset/Videos/p9i3/p9i3_Vi...,p9i3,p9i3_Video2,1.0,video2
326,/mnt/data/StressID_dataset/Videos/h8s1/h8s1_Br...,h8s1,h8s1_Breathing,0.0,breathing


In [12]:
df_complete['label'] = df_complete['binary-stress'].map({0.0: 'no-stress', 1.0: 'stress'})
print(f"df_complete shape: {df_complete.shape}")
df_complete.sample(3)

df_complete shape: (578, 6)


,path,subject,subject/task,binary-stress,task,label
121,/mnt/data/StressID_dataset/Videos/f6q3/f6q3_Ma...,f6q3,f6q3_Math,0.0,math,no-stress
12,/mnt/data/StressID_dataset/Videos/d4n6/d4n6_Sp...,d4n6,d4n6_Speaking,1.0,speaking,stress
652,/mnt/data/StressID_dataset/Videos/j9h8/j9h8_Vi...,j9h8,j9h8_Video2,1.0,video2,stress


#### **Vid-to-Frame extraction**
____

In [15]:
import cv2
from pathlib import Path
import pandas as pd
from tqdm import tqdm

# Base output directory
output_base = Path("/mnt/data/StressID-img-data")
output_base.mkdir(exist_ok=True, parents=True)

TARGET_FPS = 5
TARGET_INTERVAL = 1.0 / TARGET_FPS   # 0.2 seconds

# Wrap the main for loop with tqdm
for idx, row in tqdm(df_complete.iterrows(), total=len(df_complete), desc="Processing videos"):
    video_path = Path(row["path"])
    label = row["label"]
    subject = row["subject"]
    task = row["task"]

    # Output folder structure
    output_folder = output_base / label / subject / task
    output_folder.mkdir(parents=True, exist_ok=True)

    # Open video
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        print(f"Error opening video: {video_path}")
        continue

    next_timestamp = 0.0  # next frame time to save (seconds)
    saved_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # Timestamp (in seconds)
        timestamp = cap.get(cv2.CAP_PROP_POS_MSEC) / 1000.0

        # Save the next scheduled frame
        if timestamp >= next_timestamp:
            # ----- Modified line: Save as .png instead of .jpg -----
            frame_file = output_folder / f"frame_{saved_idx:04d}.png"
            cv2.imwrite(str(frame_file), frame)
            # --------------------------------------------------------

            saved_idx += 1
            next_timestamp += TARGET_INTERVAL

    cap.release()


Processing videos: 100%|██████████| 578/578 [1:38:02<00:00, 10.18s/it]


In [16]:
df_complete["binary-stress"].value_counts()

binary-stress
1.0    312
0.0    266
Name: count, dtype: int64

#### **Image dataframe**
___

In [17]:
from pathlib import Path

def find_files(start_directory, extension=None):
    base_path = Path(start_directory)
    pattern = f"*{extension}" if extension else "*"
    files = [str(p.resolve()) for p in base_path.rglob(pattern) 
             if not extension or p.suffix.lower() == extension.lower()]
    return sorted(files)

# Set your path and extension
files_path = "/mnt/data/StressID-img-data/"  # Change this
video_files_list = find_files(files_path, ".png")

print(f"Found {len(video_files_list)} png files:")
for file in video_files_list[:5]:
    print(file)   

Found 271597 png files:
/mnt/data/StressID-img-data/no-stress/2ea4/breathing/frame_0000.png
/mnt/data/StressID-img-data/no-stress/2ea4/breathing/frame_0001.png
/mnt/data/StressID-img-data/no-stress/2ea4/breathing/frame_0002.png
/mnt/data/StressID-img-data/no-stress/2ea4/breathing/frame_0003.png
/mnt/data/StressID-img-data/no-stress/2ea4/breathing/frame_0004.png


In [18]:
img_dataset_df = pd.DataFrame(video_files_list, columns=["path"])
img_dataset_df.sample(3)

,path
116661,/mnt/data/StressID-img-data/no-stress/p9i3/vid...
231277,/mnt/data/StressID-img-data/stress/j9h8/math/f...
268293,/mnt/data/StressID-img-data/stress/x1q3/video1...


In [19]:
img_dataset_df["str-label"] = img_dataset_df["path"].str.split("/").str[-4]
img_dataset_df.sample(3)

,path,str-label
116793,/mnt/data/StressID-img-data/no-stress/qw5t/bre...,no-stress
181636,/mnt/data/StressID-img-data/stress/9t6n/speaki...,stress
127825,/mnt/data/StressID-img-data/no-stress/tmvd/cou...,no-stress


In [20]:
img_dataset_df['label'] = img_dataset_df['str-label'].map({'no-stress': 0.0, 'stress': 1.0})
img_dataset_df.sample(3)

,path,str-label,label
249841,/mnt/data/StressID-img-data/stress/qw5t/video1...,stress,1.0
93074,/mnt/data/StressID-img-data/no-stress/h8r2/rel...,no-stress,0.0
235306,/mnt/data/StressID-img-data/stress/k2v7/readin...,stress,1.0


In [21]:
train_df = img_dataset_df.sample(frac=0.9, random_state=42441991)
test_df = img_dataset_df.drop(train_df.index)

print(f"shape train df: {train_df.shape}")
print(f"shape test df: {test_df.shape}")

shape train df: (244437, 3)
shape test df: (27160, 3)


In [22]:
train_df.sample(3)

,path,str-label,label
266735,/mnt/data/StressID-img-data/stress/x1q3/math/f...,stress,1.0
112568,/mnt/data/StressID-img-data/no-stress/m8g5/vid...,no-stress,0.0
32924,/mnt/data/StressID-img-data/no-stress/7h5u/bre...,no-stress,0.0


In [23]:
test_df.sample(3)

,path,str-label,label
52943,/mnt/data/StressID-img-data/no-stress/b2l8/bre...,no-stress,0.0
3883,/mnt/data/StressID-img-data/no-stress/2hpu/bre...,no-stress,0.0
204432,/mnt/data/StressID-img-data/stress/ctzy/counti...,stress,1.0


In [24]:
train_df.to_csv("/mnt/data/StressID-img-data/train.csv", sep=";", index=False)
test_df.to_csv("/mnt/data/StressID-img-data/test.csv", sep=";", index=False)
img_dataset_df.to_csv("/mnt/data/StressID-img-data/dataframe.csv", sep=";", index=False)